### Подключение PySpark, загрузка библиотек, настройка изображений

In [29]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, countDistinct, count, size, when, regexp_extract, split, lit, try_element_at

sns.set_style('darkgrid')
params = {'legend.fontsize': 'medium', 'figure.figsize': (10,8), 'figure.dpi': 100, 'axes.labelsize': 'medium', 'axes.titlesize': 'medium', 'xtick.labelsize': 'medium', 'ytick.labelsize': 'medium'}
plt.rcParams.update(params)

### Запуск сессии PySpark

In [3]:
spark = SparkSession.builder.appName('EDA Films').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/24 17:14:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 17:14:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Загрузка и обзор датасетов

In [9]:
df_movies = spark.read.csv('sp_movies.csv', header=True, inferSchema=True)
df_ratings = spark.read.csv('sp_ratings.csv', header=True, inferSchema=True)
df_tags = spark.read.csv('sp_tags.csv', header=True, inferSchema=True)

In [11]:
df_movies.show(5, truncate=False)

+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
+-------+----------------------------------+-------------------------------------------+
only showing top 5 rows


In [12]:
df_ratings.show(5, truncate=False)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
|1     |47     |5.0   |964983815|
|1     |50     |5.0   |964982931|
+------+-------+------+---------+
only showing top 5 rows


In [13]:
df_tags.show(5, truncate=False)

+------+-------+---------------+----------+
|userId|movieId|tag            |timestamp |
+------+-------+---------------+----------+
|2     |60756  |funny          |1445714994|
|2     |60756  |Highly quotable|1445714996|
|2     |60756  |will ferrell   |1445714992|
|2     |89774  |Boxing story   |1445715207|
|2     |89774  |MMA            |1445715200|
+------+-------+---------------+----------+
only showing top 5 rows


### Статистика

In [ ]:
print('Количество пользователей поставивших оценку:')
df_ratings.select(countDistinct('userId')).show()

Количество пользователей поставивших оценку:
+----------------------+
|count(DISTINCT userId)|
+----------------------+
|                   610|
+----------------------+



In [17]:
print('Количество оцененных фильмов:')
df_ratings.select(countDistinct('movieId')).show()

Количество оцененных фильмов:
+-----------------------+
|count(DISTINCT movieId)|
+-----------------------+
|                   9724|
+-----------------------+



In [19]:
print('Количество фильмов с комментариями (tags):')
df_tags.select(countDistinct('movieId')).show()

Количество фильмов с комментариями (tags):
+-----------------------+
|count(DISTINCT movieId)|
+-----------------------+
|                   1572|
+-----------------------+



In [22]:
print('Количество комментариев (уникальных) к фильмам (tags):')
df_tags.select(countDistinct('tag')).show()

Количество комментариев (уникальных) к фильмам (tags):
+-------------------+
|count(DISTINCT tag)|
+-------------------+
|               1589|
+-------------------+



### Предобработка и настройка данных

In [23]:
df1 = df_ratings.alias('df1')
df2 = df_tags.alias('df2')
df3 = df_movies.alias('df3')

In [24]:
df3.show()

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
|     11|American Presiden...|Comedy|Drama|Romance|
|     12|Dracula: Dead and...|       Comedy|Horror|
|     13|        Balto (1995)|Adventure|Animati...|
|     14|        Nixon (1995)|               Drama|
|     15|Cutthroat Island ...|Action|Adventure|...|
|     16|       Casino (1995)|         Crime|Drama|
|     17|Sen

### Извлечение года из названия

In [26]:
df3 = df3.withColumn('year', regexp_extract(df3['title'], r'\((\d{4})\)', 1))

In [27]:
df3.show(10, truncate=False)

+-------+----------------------------------+-------------------------------------------+----+
|movieId|title                             |genres                                     |year|
+-------+----------------------------------+-------------------------------------------+----+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|1995|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |1995|
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |1995|
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |1995|
|5      |Father of the Bride Part II (1995)|Comedy                                     |1995|
|6      |Heat (1995)                       |Action|Crime|Thriller                      |1995|
|7      |Sabrina (1995)                    |Comedy|Romance                             |1995|
|8      |Tom and Huck (1995)               |Adventure|Childr

### Извлечение данных о жанрах

In [30]:
from functools import reduce

split_expr = split(df3['genres'], r'\|')

for i in range(1, 11):
    df3 = df3.withColumn(
        f'genre{i}',
        try_element_at(split_expr, lit(i))
    )

genre_columns = [f'genre{i}' for i in range(1, 11)]

genre_count_expr = reduce(
    lambda a, b: a + b,
    [
        when(
            col(col_name).isNotNull() & (col(col_name) != '0'),
            1
        ).otherwise(0)
        for col_name in genre_columns
    ]
)

df3 = df3.withColumn(
    'genre_count',
    size(split_expr)
)



In [31]:
df3.show(truncate=False)

+-------+-------------------------------------+-------------------------------------------+----+---------+---------+--------+------+--------+------+------+------+------+-------+-----------+
|movieId|title                                |genres                                     |year|genre1   |genre2   |genre3  |genre4|genre5  |genre6|genre7|genre8|genre9|genre10|genre_count|
+-------+-------------------------------------+-------------------------------------------+----+---------+---------+--------+------+--------+------+------+------+------+-------+-----------+
|1      |Toy Story (1995)                     |Adventure|Animation|Children|Comedy|Fantasy|1995|Adventure|Animation|Children|Comedy|Fantasy |NULL  |NULL  |NULL  |NULL  |NULL   |5          |
|2      |Jumanji (1995)                       |Adventure|Children|Fantasy                 |1995|Adventure|Children |Fantasy |NULL  |NULL    |NULL  |NULL  |NULL  |NULL  |NULL   |3          |
|3      |Grumpier Old Men (1995)              |Com

### Анализ данных

In [33]:
rating_avg = df1.groupBy('movieId').agg(mean('rating').alias('rating_avg'))
rating_avg = rating_avg.withColumnRenamed('movieId', 'movieId_avg')
rating_avg.show()

+-----------+------------------+
|movieId_avg|        rating_avg|
+-----------+------------------+
|       1580| 3.487878787878788|
|       2366|              3.64|
|       3175|              3.58|
|       1088| 3.369047619047619|
|      32460|              4.25|
|      44022| 3.217391304347826|
|      96488|              4.25|
|       1238| 4.055555555555555|
|       1342|               2.5|
|       1591|2.6346153846153846|
|       1645| 3.411764705882353|
|       4519|3.3333333333333335|
|       2142|               2.7|
|        471|              3.55|
|       3997|1.8333333333333333|
|        833|               2.0|
|       3918|3.2777777777777777|
|       7982|              3.25|
|       1959|3.6666666666666665|
|      68135|              3.55|
+-----------+------------------+
only showing top 20 rows
